# Práctica 1: Simulación de Cartera de Autos – Datos SESA 2023

En esta práctica se busca crear una cartera sintética de seguros de automóviles utilizando supuestos realistas basados en el reporte **N1_SESA_Automóviles 2023** de la CNSF (actualizado al 9 de junio de 2025). El objetivo es simular siniestros y primas para distintos segmentos de riesgo y calcular la razón de siniestralidad (loss ratio) de la cartera.


## Obtención de supuestos de frecuencia y severidad

El reporte N1_SESA_Automóviles 2023 contiene un cuadro titulado **“Vehículos asegurados, frecuencia y costo medio por cobertura”** (página 13), donde se presentan, para el periodo 2021–2023, el número de vehículos asegurados, la frecuencia de siniestros y el costo medio por siniestro de cada cobertura. Para 2023 se tienen las siguientes cifras:

| Cobertura                     | Vehículos asegurados 2023 | Frecuencia 2023 | Costo medio 2023 (MX$) |
|------------------------------|---------------------------|-----------------|------------------------|
| Daños Materiales (DM)        | 10,972,512                | 15.90 %         | 22,023                 |
| Responsabilidad Civil Total (RC) | 13,069,877                | 8.50 %          | 17,147                 |
| Robo Total                   | 11,785,108                | 0.42 %          | 206,984               |
| Gastos Médicos a Ocupantes   | 12,279,106                | 2.40 %          | 4,864                  |

En esta simulación se utilizarán las coberturas de **Daños Materiales**, **Responsabilidad Civil** y **Robo Total**. La frecuencia de siniestros se calcula como `número de siniestros ÷ vehículos asegurados` y el costo medio corresponde al valor promedio pagado por siniestro. 


In [1]:
import numpy as np
import pandas as pd

# Parámetros de frecuencia (frecuencia anual por póliza)
freq = {
    'DM': 0.1590,   # 15.90 % para Daños Materiales
    'RC': 0.0850,   # 8.50 % para Responsabilidad Civil
    'Robo': 0.0042  # 0.42 % para Robo Total
}

# Severidad media de siniestro (MX$) para 2023
mean_severity = {
    'DM': 22023,
    'RC': 17147,
    'Robo': 206984
}

# Supuesto de la desviación logarítmica para la distribución log-normal
total_sigma = 0.6

# Cálculo de mu para la log-normal: mean = exp(mu + sigma^2/2)
mu = {cov: np.log(m) - (total_sigma**2)/2 for cov, m in mean_severity.items()}

mu


{'DM': 9.81984264077889, 'RC': 9.569578510178852, 'Robo': 12.060396774574128}

## Segmentación de la cartera

Se supone que la cartera se compone de tres segmentos de riesgo:

- **Bajo riesgo:** 30 % de las pólizas, multiplicador de 0.7 sobre la frecuencia base.
- **Estándar:** 50 % de las pólizas, multiplicador de 1.0.
- **Alto riesgo:** 20 % de las pólizas, multiplicador de 1.5.

Las severidades se generan a partir de distribuciones log‑normales con los parámetros calculados anteriormente (`mu` y `sigma`). Para cada póliza se genera un número de siniestros siguiendo una distribución de Poisson y se suma el costo de los siniestros simulados. Posteriormente se calculan las primas puras y comerciales (agregando un margen de 30 %).


In [4]:
# Definición de segmentos y multiplicadores
segments = ['Bajo riesgo', 'Estándar', 'Alto riesgo']
weights = [0.3, 0.5, 0.2]
multiplier = {
    'Bajo riesgo': 0.7,
    'Estándar': 1.0,
    'Alto riesgo': 1.5
}

# Número de pólizas a simular
n_policies = 10000

# Semilla para reproducibilidad
np.random.seed(42)

# Asignación de segmento a cada póliza
policies = pd.DataFrame()
policies['segmento'] = np.random.choice(segments, size=n_policies, p=weights)


policies.head()

,segmento
0,Estándar
1,Alto riesgo
2,Estándar
3,Estándar
4,Bajo riesgo


In [5]:
# Simulación de siniestros y montos
for cov in ['DM', 'RC', 'Robo']:
    base_lambda = freq[cov]
    # Número de siniestros por póliza según Poisson
    policies[f'num_siniestros_{cov}'] = policies['segmento'].apply(lambda s: np.random.poisson(base_lambda * multiplier[s]))
    # Costo de siniestros: suma de log-normales si hay siniestros
    costs = []
    for idx, row in policies.iterrows():
        n = row[f'num_siniestros_{cov}']
        if n > 0:
            severities = np.random.lognormal(mean=mu[cov], sigma=total_sigma, size=n)
            costs.append(severities.sum())
        else:
            costs.append(0.0)
    policies[f'monto_siniestros_{cov}'] = costs
    
policies.head()


,segmento,num_siniestros_DM,monto_siniestros_DM,num_siniestros_RC,monto_siniestros_RC,num_siniestros_Robo,monto_siniestros_Robo
0,Estándar,0,0.0,0,0.0,0,0.0
1,Alto riesgo,0,0.0,0,0.0,0,0.0
2,Estándar,0,0.0,0,0.0,0,0.0
3,Estándar,0,0.0,0,0.0,0,0.0
4,Bajo riesgo,0,0.0,0,0.0,0,0.0


In [6]:
# Cálculo de primas puras y técnica
margin = 0.30
for cov in ['DM', 'RC', 'Robo']:
    policies[f'prima_pura_{cov}'] = freq[cov] * mean_severity[cov] * policies['segmento'].map(multiplier)
    policies[f'prima_tecnica_{cov}'] = policies[f'prima_pura_{cov}'] * (1 + margin)

# Cálculo de totales y razón de siniestralidad
policies['monto_siniestros_total'] = policies[[f'monto_siniestros_{c}' for c in ['DM','RC','Robo']]].sum(axis=1)
policies['prima_tecnica_total'] = policies[[f'prima_tecnica_{c}' for c in ['DM','RC','Robo']]].sum(axis=1)

# Razón de siniestralidad (Loss Ratio)
policies['loss_ratio'] = policies['monto_siniestros_total'] / policies['prima_tecnica_total']

policies.head()


,segmento,num_siniestros_DM,monto_siniestros_DM,num_siniestros_RC,monto_siniestros_RC,num_siniestros_Robo,monto_siniestros_Robo,prima_pura_DM,prima_tecnica_DM,prima_pura_RC,prima_tecnica_RC,prima_pura_Robo,prima_tecnica_Robo,monto_siniestros_total,prima_tecnica_total,loss_ratio
0,Estándar,0,0.0,0,0.0,0,0.0,3501.6570,4552.15410,1457.4950,1894.74350,869.33280,1130.132640,0.0,7577.030240,0.0
1,Alto riesgo,0,0.0,0,0.0,0,0.0,5252.4855,6828.23115,2186.2425,2842.11525,1303.99920,1695.198960,0.0,11365.545360,0.0
2,Estándar,0,0.0,0,0.0,0,0.0,3501.6570,4552.15410,1457.4950,1894.74350,869.33280,1130.132640,0.0,7577.030240,0.0
3,Estándar,0,0.0,0,0.0,0,0.0,3501.6570,4552.15410,1457.4950,1894.74350,869.33280,1130.132640,0.0,7577.030240,0.0
4,Bajo riesgo,0,0.0,0,0.0,0,0.0,2451.1599,3186.50787,1020.2465,1326.32045,608.53296,791.092848,0.0,5303.921168,0.0


In [7]:
# Resumen por segmento
summary = policies.groupby('segmento').agg({
    'num_siniestros_DM': 'mean',
    'num_siniestros_RC': 'mean',
    'num_siniestros_Robo': 'mean',
    'monto_siniestros_total': 'mean',
    'prima_tecnica_total': 'mean',
    'loss_ratio': 'mean'
})
summary


,num_siniestros_DM,num_siniestros_RC,num_siniestros_Robo,monto_siniestros_total,prima_tecnica_total,loss_ratio
segmento,,,,,,
Alto riesgo,0.248216,0.124873,0.009684,9599.506312,11365.545360,0.844615
Bajo riesgo,0.116279,0.059613,0.002293,4226.260758,5303.921168,0.796818
Estándar,0.163089,0.076028,0.003811,5693.466097,7577.030240,0.751411


In [8]:
# Guardar la cartera simulada en un archivo CSV
output_path = 'cartera_autos_simulada_realista.csv'
selected_columns = ['segmento',
                    'num_siniestros_DM','num_siniestros_RC','num_siniestros_Robo',
                    'monto_siniestros_DM','monto_siniestros_RC','monto_siniestros_Robo',
                    'prima_pura_DM','prima_pura_RC','prima_pura_Robo',
                    'prima_tecnica_DM','prima_tecnica_RC','prima_tecnica_Robo',
                    'prima_tecnica_total','loss_ratio']

policies[selected_columns].to_csv(output_path, index=False)

output_path


'cartera_autos_simulada_realista.csv'

## Conclusiones

- Se utilizaron las cifras de **vehículos asegurados, frecuencia y costo medio** del cuadro de la página 13 del reporte N1_SESA_Automóviles 2023 para derivar las frecuencias y severidades promedio por cobertura
- Las frecuencias de siniestros fueron de 15.9 % para Daños Materiales, 8.5 % para Responsabilidad Civil y 0.42 % para Robo.  
- Los costos medios por siniestro se asumieron como MX$ 22 023 para Daños Materiales, MX$ 17 147 para Responsabilidad Civil y MX$ 206 984 para Robo; se desconocía la unidad exacta, por lo que se tomaron tal cual.
- La severidad se modeló con una distribución log‑normal ajustada para cada cobertura, con parámetros `mu` y `sigma` calculados a partir del costo medio y un coeficiente de variación supuesto del 0.5.
- La simulación consideró un **margen de gastos/utilidad del 30 %** y tres segmentos de riesgo con diferentes multiplicadores sobre la frecuencia base.
- La **razón de siniestralidad (loss ratio)** promedio obtenida fue cercana a 0.8, un poco superior al 75 % de referencia; se pueden ajustar las frecuencias o el margen para aproximarse al objetivo.

El archivo `cartera_autos_simulada_realista.csv` contiene la cartera simulada detallada para cada póliza.
